In [1]:
# ============================================================
# MatricMath Intelligence
# Notebook 01: Exam Document Inventory
# ============================================================
# Purpose:
# Establish the document control register before any bulk download.
# Output:
# data/metadata/exam_document_register.csv
# ============================================================

print("Notebook 01 – Exam Document Inventory")

Notebook 01 – Exam Document Inventory


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
META_DIR = DATA_DIR / "metadata"

EXAMS_DIR = RAW_DIR / "exams"
MEMOS_DIR = RAW_DIR / "memos"
DIAG_DIR = RAW_DIR / "diagnostic_reports"
CAPS_DIR = RAW_DIR / "caps"

for directory in [EXAMS_DIR, MEMOS_DIR, DIAG_DIR, CAPS_DIR, META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Folder structure ready")
print(f"Register path: {META_DIR / 'exam_document_register.csv'}")

Folder structure ready
Register path: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv


## Source Hierarchy

### Tier 1 — Official primary
- DBE examination papers
- DBE marking guidelines
- DBE diagnostic reports
- DBE CAPS documents
- DBE examination guidelines
- Umalusi reports

### Tier 2 — Official secondary
- Provincial education repositories

### Tier 3 — Backfill only
- Trusted educational archives, used only when Tier 1/2 are unavailable

Rule: Tier 1 first.

In [3]:
REGISTER_COLUMNS = [
    "document_id",
    "year",
    "exam_session",
    "paper",
    "document_type",
    "subject",
    "language",
    "source",
    "source_tier",
    "source_url",
    "file_name",
    "file_path",
    "curriculum_regime",
    "text_extractable",
    "ocr_required",
    "collection_status",
    "priority",
    "notes",
    "date_added",
]

print(f"Schema columns: {len(REGISTER_COLUMNS)}")

Schema columns: 19


In [4]:
def curriculum_regime(year: int) -> str:
    """
    Grade 12 / Matric Mathematics curriculum-regime classification.

    CAPS implementation:
    - 2012: Grade 10 starts CAPS
    - 2013: Grade 11 starts CAPS
    - 2014: Grade 12 / matric first fully assessed under CAPS
    """
    if year < 2012:
        return "pre-CAPS"
    if year in (2012, 2013):
        return "CAPS-transition"
    return "CAPS-Grade12"


def priority_for_year(year: int) -> str:
    """
    Collection priority.
    CAPS-Grade12 years are prioritised; recent years first.
    """
    if year >= 2023:
        return "high"
    if year >= 2018:
        return "medium"
    if year >= 2014:
        return "medium"   # CAPS-Grade12, still useful
    return "low"          # pre-CAPS / transition for later expansion


for year in [2010, 2012, 2013, 2014, 2018, 2024]:
    print(year, "→", curriculum_regime(year), "|", priority_for_year(year))

2010 → pre-CAPS | low
2012 → CAPS-transition | low
2013 → CAPS-transition | low
2014 → CAPS-Grade12 | medium
2018 → CAPS-Grade12 | medium
2024 → CAPS-Grade12 | high


In [5]:
records = []
now = datetime.now(timezone.utc).isoformat()

# Primary CAPS-Grade12 analytical window
YEARS = list(range(2014, 2026))

EXAM_SESSIONS = ["Nov", "May-June", "Supplementary", "Feb-March"]
PAPERS = ["P1", "P2"]
DOCUMENT_TYPES = ["exam", "memo"]

for year in YEARS:
    for session in EXAM_SESSIONS:
        for paper in PAPERS:
            for document_type in DOCUMENT_TYPES:
                document_id = (
                    f"{year}_{session}_{paper}_{document_type}_maths"
                    .lower()
                    .replace("-", "_")
                )
                records.append({
                    "document_id": document_id,
                    "year": year,
                    "exam_session": session,
                    "paper": paper,
                    "document_type": document_type,
                    "subject": "Mathematics",
                    "language": "Unknown",
                    "source": "DBE",
                    "source_tier": 1,
                    "source_url": "",
                    "file_name": "",
                    "file_path": "",
                    "curriculum_regime": curriculum_regime(year),
                    "text_extractable": "unknown",
                    "ocr_required": "unknown",
                    "collection_status": "queued",
                    "priority": priority_for_year(year),
                    "notes": "Inventory candidate; availability must be verified",
                    "date_added": now,
                })

# Diagnostic reports
for year in YEARS:
    records.append({
        "document_id": f"{year}_diagnostic_maths",
        "year": year,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "diagnostic",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": curriculum_regime(year),
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": priority_for_year(year),
        "notes": "DBE Mathematics diagnostic report target",
        "date_added": now,
    })

# Supporting curriculum documents
records.extend([
    {
        "document_id": "caps_mathematics_current",
        "year": 0,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "caps",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": "high",
        "notes": "CAPS Mathematics curriculum reference",
        "date_added": now,
    },
    {
        "document_id": "nsc_mathematics_exam_guideline",
        "year": 0,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "guideline",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": "high",
        "notes": "NSC Mathematics examination guideline",
        "date_added": now,
    },
])

register_df = pd.DataFrame(records, columns=REGISTER_COLUMNS)

print(f"Seed inventory rows: {len(register_df)}")
print("\nCurriculum regimes:")
print(register_df["curriculum_regime"].value_counts())
print("\nPriority distribution:")
print(register_df["priority"].value_counts())
display(register_df.head(15))

Seed inventory rows: 206

Curriculum regimes:
curriculum_regime
CAPS-Grade12    206
Name: count, dtype: int64

Priority distribution:
priority
medium    153
high       53
Name: count, dtype: int64


,document_id,year,exam_session,paper,document_type,subject,language,source,source_tier,source_url,file_name,file_path,curriculum_regime,text_extractable,ocr_required,collection_status,priority,notes,date_added
0,2014_nov_p1_exam_maths,2014,Nov,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
1,2014_nov_p1_memo_maths,2014,Nov,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
2,2014_nov_p2_exam_maths,2014,Nov,P2,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
3,2014_nov_p2_memo_maths,2014,Nov,P2,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
4,2014_may_june_p1_exam_maths,2014,May-June,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
5,2014_may_june_p1_memo_maths,2014,May-June,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
6,2014_may_june_p2_exam_maths,2014,May-June,P2,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
7,2014_may_june_p2_memo_maths,2014,May-June,P2,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
8,2014_supplementary_p1_exam_maths,2014,Supplementary,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
9,2014_supplementary_p1_memo_maths,2014,Supplementary,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00


In [6]:
assert list(register_df.columns) == REGISTER_COLUMNS
assert register_df["document_id"].is_unique
assert register_df["collection_status"].isin(
    ["queued", "downloaded", "verified", "missing"]
).all()
assert register_df["source_tier"].isin([1, 2, 3]).all()

# CAPS-Grade12 check for 2014+
mask_2014_plus = register_df["year"] >= 2014
assert (
    register_df.loc[mask_2014_plus, "curriculum_regime"] == "CAPS-Grade12"
).all()

print("Validation passed")

Validation passed


In [7]:
print("DOCUMENT INVENTORY SUMMARY")
print("=" * 60)
print(f"Total records        : {len(register_df)}")
print(f"Exam                 : {(register_df['document_type'] == 'exam').sum()}")
print(f"Memo                 : {(register_df['document_type'] == 'memo').sum()}")
print(f"Diagnostic           : {(register_df['document_type'] == 'diagnostic').sum()}")
print(f"Supporting           : {register_df['document_type'].isin(['caps','guideline']).sum()}")
print(f"High priority        : {(register_df['priority'] == 'high').sum()}")
print(f"Medium priority      : {(register_df['priority'] == 'medium').sum()}")
print(f"Low priority         : {(register_df['priority'] == 'low').sum()}")
print(f"CAPS-Grade12 rows    : {(register_df['curriculum_regime'] == 'CAPS-Grade12').sum()}")

DOCUMENT INVENTORY SUMMARY
Total records        : 206
Exam                 : 96
Memo                 : 96
Diagnostic           : 12
Supporting           : 2
High priority        : 53
Medium priority      : 153
Low priority         : 0
CAPS-Grade12 rows    : 206


In [8]:
exam_only = register_df[register_df["document_type"] == "exam"].copy()
session_matrix = pd.crosstab(exam_only["year"], exam_only["exam_session"])
display(session_matrix)

print("\nNote: matrix shows candidate inventory, not confirmed availability.")

exam_session,Feb-March,May-June,Nov,Supplementary
year,,,,
2014,2,2,2,2
2015,2,2,2,2
2016,2,2,2,2
2017,2,2,2,2
2018,2,2,2,2
2019,2,2,2,2
2020,2,2,2,2
2021,2,2,2,2
2022,2,2,2,2



Note: matrix shows candidate inventory, not confirmed availability.


In [9]:
register_path = META_DIR / "exam_document_register.csv"
register_df.to_csv(register_path, index=False)

print(f"Register saved → {register_path}")
print(f"Rows: {len(register_df)}")
print("NO BULK PDF DOWNLOAD PERFORMED")

Register saved → c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv
Rows: 206
NO BULK PDF DOWNLOAD PERFORMED


In [10]:
reloaded = pd.read_csv(register_path)

assert len(reloaded) == len(register_df)
assert list(reloaded.columns) == REGISTER_COLUMNS
assert reloaded["document_id"].is_unique

print("CSV reload verification passed")

CSV reload verification passed


In [11]:
# ============================================================
# LOAD CONTROL REGISTER
# ============================================================

register_path = META_DIR / "exam_document_register.csv"
register_df = pd.read_csv(register_path)

print(f"Loaded register: {register_path}")
print(f"Rows: {len(register_df)}")
print(register_df["collection_status"].value_counts())

Loaded register: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv
Rows: 206
collection_status
queued    206
Name: count, dtype: int64


In [12]:
# ============================================================
# OFFICIAL DBE AVAILABILITY AUDIT
# ============================================================
# "verified" means the document was confirmed on an official
# DBE landing page. It does NOT mean the PDF was downloaded.

VERIFIED_DOCUMENTS = [
    # 2025 November
    (2025, "Nov", "P1", "exam"),
    (2025, "Nov", "P1", "memo"),
    (2025, "Nov", "P2", "exam"),
    (2025, "Nov", "P2", "memo"),

    # 2024 November
    (2024, "Nov", "P1", "exam"),
    (2024, "Nov", "P1", "memo"),
    (2024, "Nov", "P2", "exam"),
    (2024, "Nov", "P2", "memo"),

    # 2023 November
    (2023, "Nov", "P1", "exam"),
    (2023, "Nov", "P1", "memo"),
    (2023, "Nov", "P2", "exam"),
    (2023, "Nov", "P2", "memo"),
]

verified_set = set(VERIFIED_DOCUMENTS)

updated = 0
for idx, row in register_df.iterrows():
    key = (
        int(row["year"]) if pd.notna(row["year"]) else None,
        row["exam_session"],
        row["paper"],
        row["document_type"],
    )
    if key in verified_set:
        register_df.loc[idx, "collection_status"] = "verified"
        register_df.loc[idx, "source"] = "DBE"
        register_df.loc[idx, "source_tier"] = 1
        updated += 1

print("Official availability audit completed.")
print(f"Rows updated to verified: {updated}")
print(f"Verified exam/memo targets: {len(VERIFIED_DOCUMENTS)}")

Official availability audit completed.
Rows updated to verified: 12
Verified exam/memo targets: 12


In [14]:
register_df["source_url"] = register_df["source_url"].astype("object")

mask = (
    (register_df["collection_status"] == "verified")
    & (register_df["document_type"].isin(["exam", "memo"]))
    & (register_df["year"].isin([2023, 2024, 2025]))
)

register_df.loc[mask, "source_url"] = register_df.loc[mask, "year"].map(DBE_SOURCE_PAGES)

print("URLs assigned:", mask.sum())

URLs assigned: 12
